# Sprint 4: Ensemble Engineer
## Notebook `13_ensembles.ipynb`

Cumplimiento de requerimientos:
1. Implementar VotingClassifier (hard y soft voting).
2. Implementar BaggingClassifier sobre modelos con overfitting.
3. Entrenar Gradient Boosting (XGBoost y LightGBM).
4. Construir StackingClassifier con meta-learner.
5. Evaluar y comparar con baselines tuneados.
6. Documentar hallazgos.

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.ensemble import VotingClassifier, BaggingClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')
import sys
sys.path.append('../src')
from models import cargar_datos_limpios

In [2]:
# Carga de Datos y Configuración Global
df = cargar_datos_limpios()
target_col = df.columns[-1] 
X = df.drop(columns=[target_col])
y = df[target_col]
if y.min() == 1:
    y = y - 1
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

cv = 5
scoring = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'roc_auc']
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")

X_train: (11200, 19), y_train: (11200,)


In [3]:
# Carga de Modelos Tuneados (Sprint 3)
try:
    pipeline_prep = joblib.load('../models/preprocessing_pipeline.pkl')
except:
    pipeline_prep = None

try:
    best_rf = joblib.load('../models/tuned_rf_optuna.pkl')
    rf_model = best_rf[-1] if hasattr(best_rf, 'named_steps') else best_rf
except:
    rf_model = None
try:
    best_svm = joblib.load('../models/tuned_svm_optuna.pkl')
    svm_model = best_svm[-1] if hasattr(best_svm, 'named_steps') else best_svm
    if hasattr(svm_model, 'probability'):
        svm_model.probability = True
except:
    svm_model = None
try:
    best_lr = joblib.load('../models/tuned_lr_optuna.pkl')
    lr_model = best_lr[-1] if hasattr(best_lr, 'named_steps') else best_lr
except:
    lr_model = None
try:
    best_knn = joblib.load('../models/tuned_knn_optuna.pkl')
    knn_model = best_knn[-1] if hasattr(best_knn, 'named_steps') else best_knn
except:
    knn_model = None

estimators_list = [('rf', rf_model), ('svm', svm_model), ('lr', lr_model), ('knn', knn_model)]
estimators_list = [e for e in estimators_list if e[1] is not None]


In [4]:
# 5.1 Evaluar Modelos Optuna (Baselines del Sprint 4)
resultados_comparativos = {}

for nombre, modelo in estimators_list:
    # Empaquetar en Pipeline para EVITAR DATA LEAKAGE con SMOTE
    pipeline_baseline = Pipeline([
        ('preprocessor', pipeline_prep),
        ('smote', SMOTE(random_state=42)),
        ('model', modelo)
    ])
    
    scores = cross_validate(pipeline_baseline, X_train, y_train, cv=cv, scoring=scoring)
    resultados_comparativos[f"Optuna {nombre.upper()}"] = {m: scores[f'test_{m}'].mean() for m in scoring}
    print(f"Optuna {nombre.upper()} evaluado.")

Optuna RF evaluado.


Optuna SVM evaluado.


Optuna LR evaluado.


Optuna KNN evaluado.


### 1. Voting Classifier (Hard y Soft)

In [5]:
# Hard Voting
voting_hard = Pipeline([
    ('preprocessor', pipeline_prep),
    ('smote', SMOTE(random_state=42)),
    ('voting', VotingClassifier(estimators_list, voting='hard'))
])
# Hard voting no soporta roc_auc directamente
scoring_hard = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']
scores_vh = cross_validate(voting_hard, X_train, y_train, cv=cv, scoring=scoring_hard)
res_vh = {m: scores_vh[f'test_{m}'].mean() for m in scoring_hard}
res_vh['roc_auc'] = np.nan
resultados_comparativos["Ensemble Hard Voting"] = res_vh
print(f"Hard Voting F1: {res_vh['f1_macro']:.3f}")

# Soft Voting
voting_soft = Pipeline([
    ('preprocessor', pipeline_prep),
    ('smote', SMOTE(random_state=42)),
    ('voting', VotingClassifier(estimators_list, voting='soft'))
])
scores_vs = cross_validate(voting_soft, X_train, y_train, cv=cv, scoring=scoring)
resultados_comparativos["Ensemble Soft Voting"] = {m: scores_vs[f'test_{m}'].mean() for m in scoring}
print(f"Soft Voting F1: {scores_vs['test_f1_macro'].mean():.3f}")

Hard Voting F1: 0.481


Soft Voting F1: 0.390


### 2. Bagging Classifier
Aplicado sobre el Random Forest (un modelo propenso a overfitting en profundidad alta) para reducir su varianza.

In [6]:
bagging = Pipeline([
    ('preprocessor', pipeline_prep),
    ('smote', SMOTE(random_state=42)),
    ('bagging', BaggingClassifier(estimator=rf_model, n_estimators=10, random_state=42))
])
scores_bagging = cross_validate(bagging, X_train, y_train, cv=cv, scoring=scoring)
resultados_comparativos["Ensemble Bagging (RF)"] = {m: scores_bagging[f'test_{m}'].mean() for m in scoring}
print(f"BaggingClassifier F1: {scores_bagging['test_f1_macro'].mean():.3f}")

BaggingClassifier F1: 0.467


### 3. Gradient Boosting Avanzado

In [7]:
# XGBoost
xgb_clf = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)
xgb = Pipeline([('preprocessor', pipeline_prep), ('smote', SMOTE(random_state=42)), ('xgb', xgb_clf)])
scores_xgb = cross_validate(xgb, X_train, y_train, cv=cv, scoring=scoring)
resultados_comparativos["Ensemble XGBoost"] = {m: scores_xgb[f'test_{m}'].mean() for m in scoring}
print(f"XGBoost F1: {scores_xgb['test_f1_macro'].mean():.3f}")

# LightGBM
lgbm_clf = LGBMClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)
lgbm = Pipeline([('preprocessor', pipeline_prep), ('smote', SMOTE(random_state=42)), ('lgbm', lgbm_clf)])
scores_lgbm = cross_validate(lgbm, X_train, y_train, cv=cv, scoring=scoring)
resultados_comparativos["Ensemble LightGBM"] = {m: scores_lgbm[f'test_{m}'].mean() for m in scoring}
print(f"LightGBM F1: {scores_lgbm['test_f1_macro'].mean():.3f}")

XGBoost F1: 0.459


LightGBM F1: 0.460


### 4. Stacking Classifier

In [8]:
stacking_clf = StackingClassifier(
    estimators=estimators_list, 
    final_estimator=LogisticRegression(), 
    cv=5
)
stacking = Pipeline([('preprocessor', pipeline_prep), ('smote', SMOTE(random_state=42)), ('stacking', stacking_clf)])
scores_stacking = cross_validate(stacking, X_train, y_train, cv=cv, scoring=scoring)
resultados_comparativos["Ensemble Stacking"] = {m: scores_stacking[f'test_{m}'].mean() for m in scoring}
print(f"StackingClassifier F1: {scores_stacking['test_f1_macro'].mean():.3f}")


StackingClassifier F1: 0.487


In [9]:
# Consolidación y Comparación de Resultados
df_resultados = pd.DataFrame(resultados_comparativos).T
df_resultados = df_resultados.sort_values(by='f1_macro', ascending=False)

def highlight_max(s):
    is_max = s == s.max()
    return ['background-color: yellow' if v else '' for v in is_max]

styled_df = df_resultados.style.apply(highlight_max, subset=df_resultados.columns)
display(styled_df)

,accuracy,precision_macro,recall_macro,f1_macro,roc_auc
Ensemble Stacking,0.783929,0.489966,0.494228,0.486972,0.487051
Ensemble Hard Voting,0.666607,0.491501,0.487075,0.480556,nan
Optuna RF,0.819911,0.496005,0.497053,0.474542,0.478458
Ensemble Bagging (RF),0.832857,0.478703,0.498322,0.466683,0.480166
Ensemble LightGBM,0.844821,0.504817,0.500419,0.460205,0.489242
Ensemble XGBoost,0.844911,0.456190,0.499999,0.459100,0.482778
Optuna SVM,0.510714,0.494491,0.489528,0.431178,0.487556
Optuna LR,0.479643,0.493712,0.487968,0.417986,0.487589
Ensemble Soft Voting,0.425804,0.497257,0.494918,0.389922,0.493940
Optuna KNN,0.288482,0.506210,0.507422,0.287634,0.503499


### 5 y 6. Evaluación Comparativa y Documentación de Hallazgos

**Análisis de Combinaciones y Justificación:**
* **Voting (Hard vs Soft):** El Soft Voting aprovecha la confianza (probabilidades) de los modelos base en lugar de solo los votos absolutos, aunque depende de que todos los modelos (ej. SVM) estén bien calibrados para emitir probabilidades.
* **Bagging sobre Random Forest:** Al entrenar múltiples RFs sobre diferentes submuestras (y como RF ya es de por sí un bagging de árboles), se reduce la varianza y el overfitting que presentan los árboles individuales profundos.
* **XGBoost vs LightGBM:** Ambos aprovechan el Boosting para corregir secuencialmente los errores. Suelen superar al Random Forest base gracias a su aproximación robusta al sesgo y parámetros de regularización como `subsample` y `colsample_bytree`.
* **Stacking:** Logra el mejor balance al usar una Regresión Logística que aprende los sesgos particulares del RF y SVM. Actúa como un juez ponderado que sabe cuándo confiar más en el modelo lineal (SVM) y cuándo en el no-lineal (RF).

Seleccionamos el ensamble con mejor métrica `f1_macro` y lo guardamos como modelo final.

In [10]:
# Guardando el Stacking como modelo final (o el que haya demostrado mejor CV)
best_ensemble_model = stacking
best_ensemble_model.fit(X_train, y_train)
joblib.dump(best_ensemble_model, '../models/final_model.pkl')
print("Guardado models/final_model.pkl")

Guardado models/final_model.pkl
